# Colab 02 - Verify Data + Qdrant Cloud

Use this notebook to verify data setup and Qdrant Cloud collections.


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[qdrant,agent,eval]"
!pip install requests gdown


In [ ]:
# Load Colab Secrets. Add these in Colab left sidebar > Secrets.
import os
try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("QDRANT_TEXT_COLLECTION", "text_chunks_prod")
os.environ.setdefault("QDRANT_IMAGE_COLLECTION", "image_patches_prod")
os.environ.setdefault("QDRANT_INDEX_STATE", "/content/outputs/index_state/colab_index_state.json")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))


In [ ]:
# Data setup for Colab: prefer mounted Drive, fallback to public Google Drive folder via gdown.
from pathlib import Path
import os
import shutil

PROJECT_DATA = Path("/content/project-ks2/data")
PROJECT_DATA.mkdir(parents=True, exist_ok=True)

# Optional: if you mounted Google Drive and copied data there, update this path.
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/KS_Project_2/data"),
    Path("/content/drive/MyDrive/project-ks2/data"),
]
selected = None
for candidate in DRIVE_CANDIDATES:
    if candidate.exists() and ((candidate / "eval_cases.json").exists() or (candidate / "bioasq").exists()):
        selected = candidate
        break

if selected:
    print("Copying data from Drive:", selected)
    for item in selected.iterdir():
        target = PROJECT_DATA / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
else:
    print("Drive data not found. Downloading public Drive folder with gdown.")
    !pip install -q gdown
    !rm -rf /content/project-ks2/data
    !mkdir -p /content/project-ks2/data
    !gdown --folder "https://drive.google.com/drive/folders/19tQuUoHkK6wA1IfEMVCxbkwxS-DNL5dS" -O /content/project-ks2/data --remaining-ok

# Fix common nested folder layouts.
root = Path("/content/project-ks2/data")
candidates = [root / "data", root / "KS_Project_2" / "data"]
for candidate in candidates:
    if candidate.exists() and ((candidate / "eval_cases.json").exists() or (candidate / "bioasq").exists()):
        tmp = Path("/content/project-ks2/data_fixed")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(candidate, tmp)
        shutil.rmtree(root)
        tmp.rename(root)
        break

print("Final data preview:")
!find /content/project-ks2/data -maxdepth 3 -type f | head -50


In [ ]:
# Audit data before indexing/evaluation. Raw image paths should exist; processed image paths may be normalized later by loaders.
!python -m medical_rag audit-data --data-dir data || true
!python -m medical_rag status-report --data-dir data || true


In [ ]:
!python scripts/colab_workflow.py test-env
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Optional local in-memory dry run with mock encoders.
!python -m medical_rag build-qdrant-index --qdrant-url :memory: --limit 20 --use-mock-models
